# 03_statement_tone_and_regressions

Force repo root

In [17]:
from pathlib import Path
import os

p = Path.cwd().resolve()
while not (p / ".git").exists() and p != p.parent:
    p = p.parent

os.chdir(p)
print("CWD set to repo root:", Path.cwd())


CWD set to repo root: C:\Users\Andre\iCloud Drive_new\iCloudDrive\Desktop\Thesis\thesis-vrp-fomc


Load meeting dataset (VRP + surprise)

In [18]:
import pandas as pd
import numpy as np

meet = pd.read_parquet("data/processed/meetings_vrp_usmpd.parquet")

meet["Date"] = pd.to_datetime(meet["Date"])
meet["event_date"] = pd.to_datetime(meet["event_date"])

print("Meetings rows:", len(meet))
meet[["Date","event_date","vrp","surprise_pc1"]].head()


Meetings rows: 78


,Date,event_date,vrp,surprise_pc1
0,2016-03-16,2016-03-16,0.012557,-3.888974
1,2016-04-27,2016-04-27,0.006326,0.418774
2,2016-06-15,2016-06-15,0.005135,-2.136843
3,2016-07-27,2016-07-27,0.013627,0.216385
4,2016-09-21,2016-09-21,0.009939,-1.252957


Download and cache statement HTML pages (Fed press release pages)

In [19]:
import requests
from pathlib import Path
from tqdm.auto import tqdm

RAW_HTML_DIR = Path("data/raw/fomc_statements_html")
RAW_HTML_DIR.mkdir(parents=True, exist_ok=True)

def statement_url(date, suffix="a"):
    ymd = pd.to_datetime(date).strftime("%Y%m%d")
    return f"https://www.federalreserve.gov/newsevents/pressreleases/monetary{ymd}{suffix}.htm"

headers = {"User-Agent": "Mozilla/5.0"}

download_log = []

for d in tqdm(meet["Date"].unique(), desc="Downloading statements"):
    d = pd.to_datetime(d)
    ymd = d.strftime("%Y%m%d")
    out_html = RAW_HTML_DIR / f"{ymd}.html"

    if out_html.exists():
        download_log.append({"Date": d, "url": statement_url(d,"a"), "status": "cached"})
        continue

    ok = False
    for suffix in ["a", "b", "c"]:
        url = statement_url(d, suffix)
        try:
            r = requests.get(url, headers=headers, timeout=30)
            if r.status_code == 200 and len(r.text) > 1000:
                out_html.write_text(r.text, encoding="utf-8")
                download_log.append({"Date": d, "url": url, "status": "downloaded"})
                ok = True
                break
        except Exception as e:
            download_log.append({"Date": d, "url": url, "status": f"error: {type(e).__name__}"})

    if not ok:
        download_log.append({"Date": d, "url": statement_url(d,"a"), "status": "failed"})

dl = pd.DataFrame(download_log).sort_values(["Date","status"])
print(dl["status"].value_counts())
dl.head(10)


status
cached    78
Name: count, dtype: int64


,Date,url,status
0,2016-03-16,https://www.federalreserve.gov/newsevents/pres...,cached
1,2016-04-27,https://www.federalreserve.gov/newsevents/pres...,cached
2,2016-06-15,https://www.federalreserve.gov/newsevents/pres...,cached
3,2016-07-27,https://www.federalreserve.gov/newsevents/pres...,cached
4,2016-09-21,https://www.federalreserve.gov/newsevents/pres...,cached
5,2016-11-02,https://www.federalreserve.gov/newsevents/pres...,cached
6,2016-12-14,https://www.federalreserve.gov/newsevents/pres...,cached
7,2017-02-01,https://www.federalreserve.gov/newsevents/pres...,cached
8,2017-03-15,https://www.federalreserve.gov/newsevents/pres...,cached
9,2017-05-03,https://www.federalreserve.gov/newsevents/pres...,cached


Parse statement text robustly (line-marker extraction)

In [20]:
from bs4 import BeautifulSoup
import re

TEXT_DIR = Path("data/processed/fomc_statements_text")
TEXT_DIR.mkdir(parents=True, exist_ok=True)

START_MARKERS = [
    "Information received since the Federal Open Market Committee met",
    "Recent indicators suggest",
    "The Federal Reserve is committed",
    "The Committee seeks to achieve",
]

END_MARKERS = [
    "Implementation Note",
    "Last Update",
]

def extract_statement_text(html: str) -> str:
    soup = BeautifulSoup(html, "lxml")
    lines = [ln.strip() for ln in soup.get_text("\n").splitlines()]
    lines = [ln for ln in lines if ln]

    # find start line
    start_idx = None
    for i, ln in enumerate(lines):
        if any(ln.startswith(m) for m in START_MARKERS):
            start_idx = i
            break
    if start_idx is None:
        # fallback: return a trimmed chunk (still better than empty)
        return "\n".join(lines[:200])

    # find end line after start
    end_idx = None
    for j in range(start_idx + 1, len(lines)):
        if any(lines[j].startswith(m) for m in END_MARKERS):
            end_idx = j
            break

    if end_idx is None:
        end_idx = min(start_idx + 200, len(lines))

    stmt_lines = lines[start_idx:end_idx]
    text = " ".join(stmt_lines)
    text = re.sub(r"\s+", " ", text).strip()
    return text

records = []
for d in meet["Date"].unique():
    d = pd.to_datetime(d)
    ymd = d.strftime("%Y%m%d")
    html_path = RAW_HTML_DIR / f"{ymd}.html"
    if not html_path.exists():
        records.append({"Date": d, "ymd": ymd, "text": None, "n_words": np.nan})
        continue

    html = html_path.read_text(encoding="utf-8", errors="ignore")
    text = extract_statement_text(html)

    # save plain text (frozen for reproducibility)
    (TEXT_DIR / f"{ymd}.txt").write_text(text, encoding="utf-8")

    n_words = len(text.split())
    records.append({"Date": d, "ymd": ymd, "text": text, "n_words": n_words})

stm_text = pd.DataFrame(records).sort_values("Date")
print("Statements parsed:", stm_text["text"].notna().sum(), "out of", len(stm_text))
stm_text.head()


Statements parsed: 78 out of 78


,Date,ymd,text,n_words
0,2016-03-16,20160316,Information received since the Federal Open Ma...,571
1,2016-04-27,20160427,Information received since the Federal Open Ma...,576
2,2016-06-15,20160615,Information received since the Federal Open Ma...,540
3,2016-07-27,20160727,Information received since the Federal Open Ma...,565
4,2016-09-21,20160921,Information received since the Federal Open Ma...,603


Load LM Master Dictionary and build word sets

In [21]:
from pathlib import Path

lm_path = Path("data/raw/LM_MasterDictionary.csv")
assert lm_path.exists(), "Missing data/raw/LM_MasterDictionary.csv. Download LM Master Dictionary CSV and rename to this."

lm = pd.read_csv(lm_path)

# normalize column names
lm.columns = [c.strip().lower() for c in lm.columns]

# detect the word column
word_col_candidates = ["word", "term", "token"]
word_col = next((c for c in word_col_candidates if c in lm.columns), None)
assert word_col is not None, f"Could not find word column. Columns seen: {lm.columns.tolist()[:30]}"

def to_num(series):
    return pd.to_numeric(series, errors="coerce").fillna(0)

pos_set = set(lm.loc[to_num(lm.get("positive", 0)) > 0, word_col].astype(str).str.lower())
neg_set = set(lm.loc[to_num(lm.get("negative", 0)) > 0, word_col].astype(str).str.lower())
unc_set = set(lm.loc[to_num(lm.get("uncertainty", 0)) > 0, word_col].astype(str).str.lower())

print("LM sizes:", len(pos_set), len(neg_set), len(unc_set))


LM sizes: 347 2345 297


Compute tone and uncertainty for each statement

In [22]:
import re

token_re = re.compile(r"[a-zA-Z]+")

def score_statement(text):
    if text is None or not isinstance(text, str) or len(text) == 0:
        return pd.Series({"n_tot": np.nan, "n_pos": np.nan, "n_neg": np.nan, "n_unc": np.nan,
                          "tone_net": np.nan, "unc_share": np.nan})

    tokens = token_re.findall(text.lower())
    n_tot = len(tokens)
    if n_tot == 0:
        return pd.Series({"n_tot": 0, "n_pos": 0, "n_neg": 0, "n_unc": 0,
                          "tone_net": np.nan, "unc_share": np.nan})

    n_pos = sum(t in pos_set for t in tokens)
    n_neg = sum(t in neg_set for t in tokens)
    n_unc = sum(t in unc_set for t in tokens)

    tone_net = (n_pos - n_neg) / n_tot
    unc_share = n_unc / n_tot

    return pd.Series({"n_tot": n_tot, "n_pos": n_pos, "n_neg": n_neg, "n_unc": n_unc,
                      "tone_net": tone_net, "unc_share": unc_share})

scores = stm_text["text"].apply(score_statement)
stm_feat = pd.concat([stm_text.drop(columns=["text"]), scores], axis=1)

print(stm_feat[["tone_net","unc_share"]].describe())
stm_feat.head()


        tone_net  unc_share
count  78.000000  78.000000
mean    0.005465   0.012463
std     0.006718   0.006739
min    -0.008086   0.000000
25%     0.000000   0.008239
50%     0.004348   0.011749
75%     0.009760   0.017180
max     0.020576   0.028662


,Date,ymd,n_words,n_tot,n_pos,n_neg,n_unc,tone_net,unc_share
0,2016-03-16,20160316,571,572.0,10.0,6.0,4.0,0.006993,0.006993
1,2016-04-27,20160427,576,577.0,10.0,7.0,3.0,0.005199,0.005199
2,2016-06-15,20160615,540,543.0,9.0,11.0,4.0,-0.003683,0.007366
3,2016-07-27,20160727,565,567.0,8.0,8.0,4.0,0.000000,0.007055
4,2016-09-21,20160921,603,605.0,9.0,7.0,6.0,0.003306,0.009917


Save features:

In [23]:
Path("data/processed").mkdir(parents=True, exist_ok=True)
stm_feat.to_parquet("data/processed/fomc_statements_features.parquet", index=False)
print("Saved: data/processed/fomc_statements_features.parquet")


Saved: data/processed/fomc_statements_features.parquet


Merge tone into meeting dataset and create $\Delta$ VRP

In [24]:
daily = pd.read_parquet("data/processed/vrp_vix_daily.parquet")
daily["date"] = pd.to_datetime(daily["date"])
daily = daily.set_index("date").sort_index()

daily["vrp_lag1"] = daily["vrp"].shift(1)
daily["vix_lag1"] = daily["vix"].shift(1)

meet2 = meet.merge(stm_feat[["Date","tone_net","unc_share","n_tot","n_pos","n_neg","n_unc"]], on="Date", how="left")
meet2 = meet2.merge(daily[["vrp_lag1","vix_lag1"]].reset_index().rename(columns={"date":"event_date"}), on="event_date", how="left")

meet2["dvrp"] = meet2["vrp"] - meet2["vrp_lag1"]

print("Missing tone rows:", meet2["tone_net"].isna().sum(), "out of", len(meet2))
meet2[["event_date","vrp","dvrp","surprise_pc1","tone_net","unc_share"]].head()


Missing tone rows: 0 out of 78


,event_date,vrp,dvrp,surprise_pc1,tone_net,unc_share
0,2016-03-16,0.012557,-0.006018,-3.888974,0.006993,0.006993
1,2016-04-27,0.006326,-0.000705,0.418774,0.005199,0.005199
2,2016-06-15,0.005135,-0.001489,-2.136843,-0.003683,0.007366
3,2016-07-27,0.013627,-0.000581,0.216385,0.000000,0.007055
4,2016-09-21,0.009939,-0.006305,-1.252957,0.003306,0.009917


Save:

In [25]:
meet2.to_parquet("data/processed/meetings_vrp_usmpd_tone.parquet", index=False)
print("Saved: data/processed/meetings_vrp_usmpd_tone.parquet")


Saved: data/processed/meetings_vrp_usmpd_tone.parquet


Run the MVP regressiosn (no controls yet)

In [26]:
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col
from pathlib import Path

def ols_hc1(df, y, xcols):
    X = sm.add_constant(df[xcols].astype(float))
    yv = df[y].astype(float)
    return sm.OLS(yv, X, missing="drop").fit(cov_type="HC1")

mA = ols_hc1(meet2, "vrp", ["surprise_pc1"])
mB = ols_hc1(meet2, "vrp", ["surprise_pc1", "tone_net"])
mC = ols_hc1(meet2, "vrp", ["surprise_pc1", "unc_share"])

tab = summary_col(
    [mA, mB, mC],
    stars=True,
    float_format="%.4f",
    model_names=["VRP~Surprise", "VRP~Surprise+Tone", "VRP~Surprise+Unc"],
    info_dict={"N": lambda x: f"{int(x.nobs)}", "R2": lambda x: f"{x.rsquared:.3f}"}
)

print(tab)

Path("reports/tables").mkdir(parents=True, exist_ok=True)
with open("reports/tables/reg_vrp_surprise_tone.tex", "w", encoding="utf-8") as f:
    f.write(tab.as_latex())

print("Saved: reports/tables/reg_vrp_surprise_tone.tex")



               VRP~Surprise VRP~Surprise+Tone VRP~Surprise+Unc
--------------------------------------------------------------
const          0.0102***    0.0115            0.0115          
               (0.0035)     (0.0072)          (0.0071)        
surprise_pc1   0.0016       0.0016            0.0016          
               (0.0016)     (0.0016)          (0.0016)        
tone_net                    -0.2386                           
                            (0.8093)                          
unc_share                                     -0.1024         
                                              (0.5350)        
R-squared      0.0100       0.0127            0.0105          
R-squared Adj. -0.0030      -0.0136           -0.0158         
N              78           78                78              
R2             0.010        0.013             0.011           
Standard errors in parentheses.
* p<.1, ** p<.05, ***p<.01
Saved: reports/tables/reg_vrp_surprise_tone.tex


Quick multicollinearity check

In [27]:
corr = meet2[["surprise_pc1","tone_net","unc_share"]].corr()
print(corr)


              surprise_pc1  tone_net  unc_share
surprise_pc1      1.000000  0.010287   0.000336
tone_net          0.010287  1.000000  -0.118437
unc_share         0.000336 -0.118437   1.000000


Build pre-event controls from the daily dataset

In [28]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col
from pathlib import Path

daily = pd.read_parquet("data/processed/vrp_vix_daily.parquet")
daily["date"] = pd.to_datetime(daily["date"])
daily = daily.set_index("date").sort_index()

h = 22

# Pre-event (backward-looking) realized variance regime using past returns up to t-1
daily["rv_back_22"] = daily["ret2"].rolling(h).sum().shift(1) * (252 / h)

# Lagged implied variance (or use vix_lag1)
daily["iv_lag1"] = daily["iv"].shift(1)
daily["vix_lag1"] = daily["vix"].shift(1)

# Merge these controls onto meeting dataset by event_date
meet3 = meet2.merge(
    daily[["iv_lag1", "vix_lag1", "rv_back_22"]].reset_index().rename(columns={"date":"event_date"}),
    on="event_date",
    how="left"
)

print(meet3[["event_date","vrp","surprise_pc1","tone_net","unc_share","iv_lag1","rv_back_22"]].isna().sum())
meet3.head()


event_date      0
vrp             0
surprise_pc1    0
tone_net        0
unc_share       0
iv_lag1         0
rv_back_22      0
dtype: int64


,Date,event_date,SEP,Unscheduled,ED1,ED2,ED3,ED4,surprise_pc1,sp500,...,n_tot,n_pos,n_neg,n_unc,vrp_lag1,vix_lag1_x,dvrp,iv_lag1,vix_lag1_y,rv_back_22
0,2016-03-16,2016-03-16,1,0,-0.055,-0.080,-0.100,-0.110,-3.888974,2027.22,...,572.0,10.0,6.0,4.0,0.018575,16.84,-0.006018,0.028359,16.84,0.028988
1,2016-04-27,2016-04-27,0,0,0.005,0.010,0.010,0.015,0.418774,2095.15,...,577.0,10.0,7.0,3.0,0.007031,13.96,-0.000705,0.019488,13.96,0.009209
2,2016-06-15,2016-06-15,1,0,-0.035,-0.045,-0.050,-0.055,-2.136843,2071.50,...,543.0,9.0,11.0,4.0,0.006624,20.50,-0.001489,0.042025,20.50,0.008880
3,2016-07-27,2016-07-27,0,0,0.005,0.005,0.005,0.005,0.216385,2166.58,...,567.0,8.0,8.0,4.0,0.014209,13.05,-0.000581,0.017030,13.05,0.033398
4,2016-09-21,2016-09-21,1,0,-0.025,-0.025,-0.025,-0.030,-1.252957,2163.12,...,605.0,9.0,7.0,6.0,0.016245,15.92,-0.006305,0.025345,15.92,0.014594


Re-run regressions with controls (VRP level)

In [29]:
def ols_hc1(df, y, xcols):
    X = sm.add_constant(df[xcols].astype(float))
    yv = df[y].astype(float)
    return sm.OLS(yv, X, missing="drop").fit(cov_type="HC1")

# Choose control set: keep it minimal
controls = ["iv_lag1", "rv_back_22"]

mA_c = ols_hc1(meet3, "vrp", ["surprise_pc1"] + controls)
mB_c = ols_hc1(meet3, "vrp", ["surprise_pc1", "tone_net"] + controls)
mC_c = ols_hc1(meet3, "vrp", ["surprise_pc1", "unc_share"] + controls)

tab_c = summary_col(
    [mA_c, mB_c, mC_c],
    stars=True,
    float_format="%.4f",
    model_names=["VRP~Surprise+Ctrls", "VRP~Surprise+Tone+Ctrls", "VRP~Surprise+Unc+Ctrls"],
    info_dict={"N": lambda x: f"{int(x.nobs)}", "R2": lambda x: f"{x.rsquared:.3f}"}
)

print(tab_c)

Path("reports/tables").mkdir(parents=True, exist_ok=True)
with open("reports/tables/reg_vrp_surprise_tone_controls.tex", "w", encoding="utf-8") as f:
    f.write(tab_c.as_latex())

print("Saved: reports/tables/reg_vrp_surprise_tone_controls.tex")



               VRP~Surprise+Ctrls VRP~Surprise+Tone+Ctrls VRP~Surprise+Unc+Ctrls
--------------------------------------------------------------------------------
const          -0.0026            -0.0022                 -0.0045               
               (0.0039)           (0.0068)                (0.0089)              
surprise_pc1   0.0010             0.0010                  0.0009                
               (0.0018)           (0.0017)                (0.0018)              
iv_lag1        0.4276**           0.4268**                0.4367**              
               (0.1788)           (0.1762)                (0.1881)              
rv_back_22     -0.1048            -0.1080                 -0.1085               
               (0.1384)           (0.1452)                (0.1367)              
tone_net                          -0.0689                                       
                                  (0.8081)                                      
unc_share                  

Robustness &mdash; meeting-day change in VRP ($\Delta$ VRP)

In [30]:
mA_d = ols_hc1(meet3, "dvrp", ["surprise_pc1"] + controls)
mB_d = ols_hc1(meet3, "dvrp", ["surprise_pc1", "tone_net"] + controls)
mC_d = ols_hc1(meet3, "dvrp", ["surprise_pc1", "unc_share"] + controls)

tab_d = summary_col(
    [mA_d, mB_d, mC_d],
    stars=True,
    float_format="%.4f",
    model_names=["ΔVRP~Surprise+Ctrls", "ΔVRP~Surprise+Tone+Ctrls", "ΔVRP~Surprise+Unc+Ctrls"],
    info_dict={"N": lambda x: f"{int(x.nobs)}", "R2": lambda x: f"{x.rsquared:.3f}"}
)

print(tab_d)

with open("reports/tables/reg_dvrp_surprise_tone_controls.tex", "w", encoding="utf-8") as f:
    f.write(tab_d.as_latex())

print("Saved: reports/tables/reg_dvrp_surprise_tone_controls.tex")



               ΔVRP~Surprise+Ctrls ΔVRP~Surprise+Tone+Ctrls ΔVRP~Surprise+Unc+Ctrls
-----------------------------------------------------------------------------------
const          0.0020              0.0035                   -0.0001                
               (0.0021)            (0.0026)                 (0.0030)               
surprise_pc1   0.0005              0.0006                   0.0005                 
               (0.0010)            (0.0010)                 (0.0010)               
iv_lag1        -0.0198             -0.0225                  -0.0095                
               (0.1039)            (0.1030)                 (0.1005)               
rv_back_22     -0.0715             -0.0814                  -0.0757                
               (0.0705)            (0.0774)                 (0.0697)               
tone_net                           -0.2137                                         
                                   (0.2004)                                

Absolute instead of signed surprise

In [31]:
meet3["abs_surprise"] = meet3["surprise_pc1"].abs()

m_abs = ols_hc1(meet3, "vrp", ["abs_surprise", "tone_net", "iv_lag1", "rv_back_22"])
print(m_abs.summary())


                            OLS Regression Results                            
Dep. Variable:                    vrp   R-squared:                       0.079
Model:                            OLS   Adj. R-squared:                  0.029
Method:                 Least Squares   F-statistic:                     2.872
Date:                Thu, 12 Feb 2026   Prob (F-statistic):             0.0287
Time:                        09:50:51   Log-Likelihood:                 164.10
No. Observations:                  78   AIC:                            -318.2
Df Residuals:                      73   BIC:                            -306.4
Df Model:                           4                                         
Covariance Type:                  HC1                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const           -0.0021      0.008     -0.263   

Daily event premmium regressions using Newwey-West/HAC with 22 lags

In [32]:
import statsmodels.api as sm

daily = pd.read_parquet("data/processed/vrp_vix_daily.parquet")
daily["date"] = pd.to_datetime(daily["date"])
daily = daily.set_index("date").sort_index()

h = 22
daily["rv_back_22"] = daily["ret2"].rolling(h).sum().shift(1) * (252 / h)
daily["iv_lag1"] = daily["iv"].shift(1)

# FOMC meeting-day dummy
fomc_days = pd.to_datetime(meet3["event_date"].unique())
daily["fomc_day"] = daily.index.isin(fomc_days).astype(int)

X = sm.add_constant(daily[["fomc_day", "iv_lag1", "rv_back_22"]])
y = daily["vrp"]

m_daily = sm.OLS(y, X, missing="drop").fit(cov_type="HAC", cov_kwds={"maxlags": 22})
print(m_daily.summary())


                            OLS Regression Results                            
Dep. Variable:                    vrp   R-squared:                       0.029
Model:                            OLS   Adj. R-squared:                  0.028
Method:                 Least Squares   F-statistic:                     29.91
Date:                Thu, 12 Feb 2026   Prob (F-statistic):           5.43e-19
Time:                        09:50:51   Log-Likelihood:                 3048.0
No. Observations:                2469   AIC:                            -6088.
Df Residuals:                    2465   BIC:                            -6065.
Df Model:                           3                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0030      0.005      0.589      0.5

Make sure exported tables have proper captions/labels

In [33]:
from pathlib import Path

def save_summarycol_with_caption(tab, path, caption, label):
    tex = tab.as_latex()
    tex = tex.replace(r"\caption{}", rf"\caption{{{caption}}}")
    tex = tex.replace(r"\label{}",   rf"\label{{{label}}}")
    Path(path).write_text(tex, encoding="utf-8")

save_summarycol_with_caption(
    tab_c,
    "reports/tables/reg_vrp_surprise_tone_controls.tex",
    "Meeting-level regressions of VRP on policy surprise and statement tone/uncertainty with volatility-regime controls.",
    "tab:meet_vrp_controls"
)

save_summarycol_with_caption(
    tab_d,
    "reports/tables/reg_dvrp_surprise_tone_controls.tex",
    "Robustness: meeting-level regressions using $\\Delta VRP$ on meeting days with the same controls.",
    "tab:meet_dvrp_controls"
)


Winsorize VRP at 1% tails

Compute winsorization cutoffs from daily VRP

In [34]:
import pandas as pd
import numpy as np

p = 0.01  # 1% tails

daily = pd.read_parquet("data/processed/vrp_vix_daily.parquet")
daily["date"] = pd.to_datetime(daily["date"])
daily = daily.set_index("date").sort_index()

# Daily VRP cutoffs (stable)
lo_vrp, hi_vrp = daily["vrp"].quantile([p, 1 - p])
print("Daily VRP 1%/99% cutoffs:", lo_vrp, hi_vrp)

# Also for daily ΔVRP if you want
daily["dvrp"] = daily["vrp"] - daily["vrp"].shift(1)
lo_dvrp, hi_dvrp = daily["dvrp"].quantile([p, 1 - p])
print("Daily ΔVRP 1%/99% cutoffs:", lo_dvrp, hi_dvrp)


Daily VRP 1%/99% cutoffs: -0.19372134470424376 0.11031637971706418
Daily ΔVRP 1%/99% cutoffs: -0.037699905509438476 0.04682927685796079


Winsorize meeting VRP and $\Delta$ VRP using cutoffs

In [35]:
meet3["vrp_w"]  = meet3["vrp"].clip(lo_vrp, hi_vrp)
meet3["dvrp_w"] = meet3["dvrp"].clip(lo_dvrp, hi_dvrp)

# Quick before/after check
print(meet3[["vrp","vrp_w","dvrp","dvrp_w"]].describe(percentiles=[0.01,0.05,0.5,0.95,0.99]))


             vrp      vrp_w       dvrp     dvrp_w
count  78.000000  78.000000  78.000000  78.000000
mean    0.010177   0.010177  -0.000504  -0.001200
std     0.030957   0.030957   0.013591   0.009697
min    -0.171436  -0.171436  -0.025233  -0.025233
1%     -0.082998  -0.082998  -0.023389  -0.023389
5%     -0.025158  -0.025158  -0.012694  -0.012694
50%     0.009618   0.009618  -0.001608  -0.001608
95%     0.049005   0.049005   0.006046   0.006046
99%     0.071661   0.071661   0.066988   0.046829
max     0.109428   0.109428   0.086885   0.046829


Run the same controlled regressions with winsorized VRP

In [36]:
mA_w = ols_hc1(meet3, "vrp_w", ["surprise_pc1", "iv_lag1", "rv_back_22"])
mB_w = ols_hc1(meet3, "vrp_w", ["surprise_pc1", "tone_net", "iv_lag1", "rv_back_22"])
mC_w = ols_hc1(meet3, "vrp_w", ["surprise_pc1", "unc_share", "iv_lag1", "rv_back_22"])

from statsmodels.iolib.summary2 import summary_col
tab_w = summary_col(
    [mA_w, mB_w, mC_w],
    stars=True,
    float_format="%.4f",
    model_names=["VRP(w)~Surp+Ctrls", "VRP(w)~Surp+Tone+Ctrls", "VRP(w)~Surp+Unc+Ctrls"],
    info_dict={"N": lambda x: f"{int(x.nobs)}", "R2": lambda x: f"{x.rsquared:.3f}"}
)
print(tab_w)

from pathlib import Path
Path("reports/tables").mkdir(parents=True, exist_ok=True)
with open("reports/tables/reg_vrp_winsor1pct_controls.tex", "w", encoding="utf-8") as f:
    f.write(tab_w.as_latex())

print("Saved: reports/tables/reg_vrp_winsor1pct_controls.tex")



               VRP(w)~Surp+Ctrls VRP(w)~Surp+Tone+Ctrls VRP(w)~Surp+Unc+Ctrls
-----------------------------------------------------------------------------
const          -0.0026           -0.0022                -0.0045              
               (0.0039)          (0.0068)               (0.0089)             
surprise_pc1   0.0010            0.0010                 0.0009               
               (0.0018)          (0.0017)               (0.0018)             
iv_lag1        0.4276**          0.4268**               0.4367**             
               (0.1788)          (0.1762)               (0.1881)             
rv_back_22     -0.1048           -0.1080                -0.1085              
               (0.1384)          (0.1452)               (0.1367)             
tone_net                         -0.0689                                     
                                 (0.8081)                                    
unc_share                                               0.1270 

Winsorized $\Delta$ VRP robustness table

In [37]:
mA_dw = ols_hc1(meet3, "dvrp_w", ["surprise_pc1", "iv_lag1", "rv_back_22"])
mB_dw = ols_hc1(meet3, "dvrp_w", ["surprise_pc1", "tone_net", "iv_lag1", "rv_back_22"])
mC_dw = ols_hc1(meet3, "dvrp_w", ["surprise_pc1", "unc_share", "iv_lag1", "rv_back_22"])

tab_dw = summary_col(
    [mA_dw, mB_dw, mC_dw],
    stars=True,
    float_format="%.4f",
    model_names=["ΔVRP(w)~Surp+Ctrls", "ΔVRP(w)~Surp+Tone+Ctrls", "ΔVRP(w)~Surp+Unc+Ctrls"],
    info_dict={"N": lambda x: f"{int(x.nobs)}", "R2": lambda x: f"{x.rsquared:.3f}"}
)
print(tab_dw)

with open("reports/tables/reg_dvrp_winsor1pct_controls.tex", "w", encoding="utf-8") as f:
    f.write(tab_dw.as_latex())

print("Saved: reports/tables/reg_dvrp_winsor1pct_controls.tex")



               ΔVRP(w)~Surp+Ctrls ΔVRP(w)~Surp+Tone+Ctrls ΔVRP(w)~Surp+Unc+Ctrls
--------------------------------------------------------------------------------
const          0.0019             0.0027                  0.0001                
               (0.0017)           (0.0022)                (0.0027)              
surprise_pc1   0.0004             0.0004                  0.0004                
               (0.0008)           (0.0008)                (0.0008)              
iv_lag1        -0.0554            -0.0570                 -0.0466               
               (0.0699)           (0.0702)                (0.0694)              
rv_back_22     -0.0429            -0.0487                 -0.0465               
               (0.0438)           (0.0471)                (0.0440)              
tone_net                          -0.1244                                       
                                  (0.1373)                                      
unc_share                  

Quick check of clipped count

In [38]:
n_clip_vrp = ((meet3["vrp"] < lo_vrp) | (meet3["vrp"] > hi_vrp)).sum()
n_clip_dvrp = ((meet3["dvrp"] < lo_dvrp) | (meet3["dvrp"] > hi_dvrp)).sum()

print("Meeting VRP clipped count:", n_clip_vrp, "out of", len(meet3))
print("Meeting ΔVRP clipped count:", n_clip_dvrp, "out of", len(meet3))


Meeting VRP clipped count: 0 out of 78
Meeting ΔVRP clipped count: 2 out of 78


Add captions to the winsorized tables

In [39]:
from pathlib import Path

def add_caption_label(tex_path, caption, label):
    tex = Path(tex_path).read_text(encoding="utf-8")
    tex = tex.replace(r"\caption{}", rf"\caption{{{caption}}}")
    tex = tex.replace(r"\label{}",   rf"\label{{{label}}}")
    Path(tex_path).write_text(tex, encoding="utf-8")

add_caption_label(
    "reports/tables/reg_vrp_winsor1pct_controls.tex",
    "Robustness: Meeting-level regressions using VRP winsorized at the 1\\% tails (daily cutoffs).",
    "tab:meet_vrp_winsor"
)

add_caption_label(
    "reports/tables/reg_dvrp_winsor1pct_controls.tex",
    "Robustness: Meeting-level regressions using $\\Delta VRP$ winsorized at the 1\\% tails (daily cutoffs).",
    "tab:meet_dvrp_winsor"
)
